## What is an ERD?

An **Entity Relationship Diagram (ERD)** is a visual map of a database. It shows:

- The **tables** in the database (each table represents a type of entity, such as a customer or a film)
- The **columns** each table contains, along with their data types
- The **relationships** between tables, indicated by connecting lines and notation symbols

ERDs are used by database designers to plan structure, and by analysts to understand a database they did not design themselves. When you are writing a JOIN query, the ERD tells you which column to join on and whether the relationship is one-to-one, one-to-many, or many-to-many.

## The pagila database

The pagila database models a fictional DVD rental business — similar in structure to what Blockbuster Video's database might have looked like. It has **14 tables** across two main areas:

**Film catalogue:** `film`, `actor`, `film_actor`, `category`, `film_category`, `language`

**Business operations:** `customer`, `staff`, `store`, `address`, `city`, `country`, `inventory`, `rental`, `payment`

Relational databases serving healthcare, retail, government, and finance organisations all follow the same structural principles as pagila. As you explore this database, consider what equivalent tables might look like in your own organisation. A hospital system might have `patient`, `ward`, `appointment`, and `prescription` tables with similar patterns of foreign keys and junction tables.

## Full Entity Relationship Diagram

The diagram below shows all 14 tables, their columns, and the relationships between them. The lines indicate foreign key connections — follow a line between two tables to find the column that links them.

```mermaid
erDiagram
    country ||--|{ city : " "
    city ||--|{ address : " "
    address ||--o{ staff : " "
    address ||--o| store : " "
    address ||--o{ customer : " "
    store ||--|{ customer : " "
    store ||--|{ staff : " "
    store ||--o{ inventory : " "
    staff ||--o{ rental : " "
    staff ||--o{ payment : " "
    inventory |o--o{ rental : " "
    rental ||--o{ payment : " "
    customer ||--o{ payment : " "
    customer ||--o{ rental : " "
    actor ||--o{ film_actor : " "
    category ||--o{ film_category : " "
    language ||--|{ film : " "
    film ||--|{ film_actor : " "
    film ||--|{ film_category : " "
    film ||--o{ inventory : " "

    actor {
        integer actor_id PK
        varchar first_name
        varchar last_name
        timestamp last_update
    }
    category {
        integer category_id PK
        varchar name
        timestamp last_update
    }
    film {
        integer film_id PK
        varchar title
        text description
        year release_year
        smallint language_id FK
        smallint rental_duration
        numeric rental_rate
        smallint length
        numeric replacement_cost
        mpaa_rating rating
        timestamp last_update
    }
    film_actor {
        smallint actor_id FK
        smallint film_id FK
        timestamp last_update
    }
    film_category {
        smallint film_id FK
        smallint category_id FK
        timestamp last_update
    }
    address {
        integer address_id PK
        varchar address
        varchar address2
        varchar district
        smallint city_id FK
        varchar postal_code
        varchar phone
        timestamp last_update
    }
    city {
        integer city_id PK
        varchar city
        smallint country_id FK
        timestamp last_update
    }
    country {
        integer country_id PK
        varchar country
        timestamp last_update
    }
    customer {
        integer customer_id PK
        smallint store_id FK
        varchar first_name
        varchar last_name
        varchar email
        smallint address_id FK
        boolean activebool
        date create_date
        timestamp last_update
        integer active
    }
    inventory {
        integer inventory_id PK
        smallint film_id FK
        smallint store_id FK
        timestamp last_update
    }
    language {
        integer language_id PK
        character name
        timestamp last_update
    }
    payment {
        integer payment_id PK
        smallint customer_id FK
        smallint staff_id FK
        integer rental_id FK
        numeric amount
        timestamp payment_date
    }
    rental {
        integer rental_id PK
        timestamp rental_date
        integer inventory_id FK
        smallint customer_id FK
        timestamp return_date
        smallint staff_id FK
        timestamp last_update
    }
    staff {
        integer staff_id PK
        varchar first_name
        varchar last_name
        smallint address_id FK
        varchar email
        smallint store_id FK
        boolean active
        varchar username
        timestamp last_update
    }
    store {
        integer store_id PK
        smallint manager_staff_id FK
        smallint address_id FK
        timestamp last_update
    }
```

## Reading the diagram

The lines between tables represent relationships, using a standard notation:

| Symbol | Meaning |
|:-------|:--------|
| `\|\|` | Exactly one |
| `o\|` | Zero or one |
| `\|{` | One or more |
| `o{` | Zero or more |

For example, `country ||--|{ city` means: **one** country has **one or more** cities. A country must exist for a city to belong to it, and every country in the database has at least one city.

Columns marked **PK** are **primary keys** — unique identifiers for each row in that table. No two rows share the same primary key value. Columns marked **FK** are **foreign keys** — they store the primary key of a row in another table, creating the link between them.

## Key relationships to know

These are the four table chains you will use most often when writing JOIN queries. Each includes a template you can adapt directly in your own queries.

### Customer addresses
*(used in JOIN exercises — Notebooks 4 and 5)*

A customer's address is not stored directly in the `customer` table. Instead, `customer` holds an `address_id` that links to the `address` table, which in turn links to `city`, and then to `country`. This chain of foreign keys is a common pattern in relational databases — it avoids duplicating city and country names across thousands of customer records.

```
customer → address → city → country
```

```sql
FROM customer c
LEFT JOIN address  a  ON c.address_id  = a.address_id
LEFT JOIN city     ct ON a.city_id     = ct.city_id
LEFT JOIN country  co ON ct.country_id = co.country_id
```

### Films and actors
*(linked through the junction table `film_actor`)*

A film can have many actors, and an actor can appear in many films. This **many-to-many relationship** cannot be represented with a single foreign key — instead, the `film_actor` table acts as a bridge, holding one row for each actor-film pairing.

```
actor ← film_actor → film
```

```sql
FROM film_actor fa
LEFT JOIN actor a ON fa.actor_id = a.actor_id
LEFT JOIN film  f ON fa.film_id  = f.film_id
```

### Tracing sales back to films
*(required for stretch questions in Notebook 6)*

A payment record does not directly reference the film that was rented. To connect a payment to its film, you must trace through three tables: a film was placed in `inventory`, a copy from `inventory` was rented (creating a `rental` record), and the rental generated a `payment`.

```
film → inventory → rental → payment
```

```sql
FROM film f
INNER JOIN inventory i ON f.film_id      = i.film_id
INNER JOIN rental    r ON i.inventory_id = r.inventory_id
INNER JOIN payment   p ON r.rental_id    = p.rental_id
```

### Films and genres
*(linked through the junction table `film_category`)*

Like the actor relationship, a film can belong to multiple categories and a category can contain many films. The `film_category` junction table handles this.

```
film → film_category → category
```

```sql
FROM film f
INNER JOIN film_category fc ON f.film_id       = fc.film_id
INNER JOIN category      c  ON fc.category_id  = c.category_id
```

## Creating your own ERDs

You will encounter databases you have not seen before throughout this programme and in your career. Being able to create an ERD — or at least sketch the key relationships — is a practical skill for any data professional.

### 1. dbdiagram.io

[dbdiagram.io](https://dbdiagram.io/) is a free online tool that generates ERDs from a simple SQL-like syntax. You define your tables and relationships in the left panel; the diagram updates automatically on the right.

```
TABLE actor {
    actor_id integer [pk]
    first_name varchar
    last_name varchar
    last_update timestamp
}
```

The full pagila ERD is available at: [https://dbdiagram.io/d/Pagilla-6526d961ffbf5169f080ef33](https://dbdiagram.io/d/Pagilla-6526d961ffbf5169f080ef33)

### 2. Draw DB

[Draw DB](https://www.drawdb.app/) is another free tool for creating database diagrams with a visual interface. Rather than writing syntax, you build the diagram by adding tables and drawing connections between them directly on the canvas — useful if you prefer a point-and-click approach.

### 3. Mermaid

[Mermaid](https://mermaid.js.org/) is the diagramming format used throughout these notebooks. It works directly in Jupyter markdown cells when `jupyterlab_myst` is installed, which means your ERDs live alongside your code and queries rather than in a separate tool.

The syntax starts with `erDiagram`, defines the relationships between tables, then defines the columns of each table. Here is a minimal example showing two tables from pagila:

```
erDiagram
    customer ||--o{ payment : " "

    customer {
        integer customer_id PK
        varchar first_name
        varchar last_name
    }
    payment {
        integer payment_id PK
        smallint customer_id FK
        numeric amount
    }
```

For programmatic schema exploration — finding out what tables and columns exist in a database without an ERD — see the [SQLite Table Identification notebook](SQLite_table_identification.ipynb).

## Reflection

Take a few minutes to think about the following questions. You may want to write brief notes — these are good preparation for the workshop discussion and for your DT401 Module Challenge portfolio, where you will need to reflect on data management systems and their role in digital solutions.

**1. Your organisation's data model:** Consider a system your organisation uses — a CRM, a finance system, a case management tool, or anything that stores structured data. What tables might that system's database contain? What relationships would exist between them?

**2. Design decisions:** The pagila database stores customer addresses in a separate `address` table rather than directly in `customer`. What are the advantages of this approach? Can you think of a disadvantage?

**3. Data structure in practice:** How does understanding a database's structure change the questions you can ask of it? Think about a business question that would be difficult or impossible to answer without knowing how the tables connect.

*Write your reflections below:*

1. 

2.

3.

---

## Summary

This notebook gives you a complete map of the pagila database. Return to it whenever you need to:

- Check which columns are available in a table
- Find the foreign key that connects two tables
- Plan a JOIN chain before writing your query
- Look up the notation used in ERD relationship lines

**Next:** If you want to explore the database structure programmatically — using SQL queries rather than a diagram — see the [SQLite Table Identification notebook](SQLite_table_identification.ipynb). To start practising SQL, return to [Notebook 1: Selecting Data](1_selecting_data.ipynb).

---

**Notebook Version:** 1.1
**Last Updated:** March 2026